# 🎨 ToonForge — Real-Time Hybrid Cartoon Studio

**SceneNet (full frame):** Anime-style rendering for body, clothes, and background  
**FaceForge (face regions):** Premium face stylization with identity preservation  
**No face?** Falls back to SceneNet full-scene automatically  

> **Runtime → Change to T4 GPU** before running

---


In [ ]:
# 1. Install dependencies + clone model repos
!pip install -q gradio huggingface_hub dlib ninja

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# FaceForge backbone (VToonify-based, face-only high quality)
!git clone --depth 1 https://huggingface.co/spaces/PKUWilliamYang/VToonify /content/faceforge_backbone

# SceneNet backbone (AnimeGANv2, full-scene cartoon)
!git clone --depth 1 https://github.com/bryandlee/animegan2-pytorch.git /content/scenenet_backbone

os.chdir('/content/faceforge_backbone')

import torch
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Initialize FaceForge engine (auto-downloads weights from HuggingFace)
from vtoonify_model import Model

face_engine = Model('cuda')

print("Available face styles:")
for k in face_engine.style_types.keys():
    tag = "adjustable degree" if k.endswith("-d") else "fixed style"
    print(f"  {k:20s} ({tag})")


In [ ]:
# 3. Select face style + load weights
STYLE = 'cartoon1-d'       # see list above
STYLE_DEGREE = 0.5         # 0.0 (subtle) → 1.0 (full strength)
FACE_PADDING = [200, 200, 200, 200]  # top, bottom, left, right face crop margin

exstyle, info = face_engine.load_model(STYLE)
print(info)
mem = torch.cuda.memory_allocated() / 1024**3
print(f'GPU RAM: {mem:.1f}GB')


In [ ]:
# 4. Initialize SceneNet + webcam + hybrid pipeline
import sys
import importlib.util

# Load SceneNet generator (dynamic import avoids name clash with FaceForge's model/)
scenenet_path = '/content/scenenet_backbone/model.py'
spec = importlib.util.spec_from_file_location("scenenet_generator", scenenet_path)
scenenet_module = importlib.util.module_from_spec(spec)
sys.modules["scenenet_generator"] = scenenet_module
spec.loader.exec_module(scenenet_module)
SceneGenerator = scenenet_module.Generator

import torchvision.transforms.functional as TF

scene_weight = '/content/scenenet_backbone/weights/face_paint_512_v2.pt'
if not os.path.exists(scene_weight):
    os.makedirs('/content/scenenet_backbone/weights', exist_ok=True)
    torch.hub.download_url_to_file(
        'https://github.com/bryandlee/animegan2-pytorch/raw/main/weights/face_paint_512_v2.pt',
        scene_weight)

scene_net = SceneGenerator().eval().to('cuda')
scene_net.load_state_dict(torch.load(scene_weight, map_location='cuda'))
print('\u2705 SceneNet loaded (full-frame anime engine)')

@torch.inference_mode()
def scene_render(pil_img):
    """Render full frame through SceneNet."""
    w, h = pil_img.size
    nw, nh = max(8, (w//8)*8), max(8, (h//8)*8)
    img = pil_img.resize((nw, nh), Image.LANCZOS)
    t = TF.to_tensor(img).unsqueeze(0).to('cuda') * 2 - 1
    out = scene_net(t)
    out = ((out + 1) / 2).clamp(0, 1)
    return TF.to_pil_image(out.squeeze(0).cpu())

# --- Webcam capture ---
from IPython.display import display, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
from PIL import Image
import numpy as np
import io, time, cv2

def grab_frame():
    """Capture one frame from browser webcam."""
    d = eval_js(\"\"\"
        (async () => {
            if (!document.getElementById(\"webcam\")) {
                const c = document.createElement(\"div\"); c.id = \"cam-container\";
                document.body.appendChild(c);
                const v = document.createElement(\"video\");
                v.style.display = \"none\"; v.id = \"webcam\"; c.appendChild(v);
                const tmpStream = await navigator.mediaDevices.getUserMedia({video: true});
                tmpStream.getTracks().forEach(t => t.stop());
                const devices = await navigator.mediaDevices.enumerateDevices();
                const cams = devices.filter(d => d.kind === \"videoinput\");
                let targetCam = cams.find(c => c.label.toLowerCase().includes(\"obs\"));
                if (!targetCam) targetCam = cams[cams.length - 1];
                const constraints = {video: {deviceId: {exact: targetCam.deviceId}, width: 640, height: 480}};
                const s = await navigator.mediaDevices.getUserMedia(constraints);
                v.srcObject = s; await v.play();
                const cn = document.createElement(\"canvas\");
                cn.id = \"cap-canvas\"; cn.style.display = \"none\"; c.appendChild(cn);
                await new Promise(r => setTimeout(r, 500));
            }
            const v = document.getElementById(\"webcam\");
            const c = document.getElementById(\"cap-canvas\");
            const maxW = 640, maxH = 480;
            let cw = v.videoWidth, ch = v.videoHeight;
            if (cw > maxW || ch > maxH) {
                const s = Math.min(maxW/cw, maxH/ch);
                cw = Math.floor(cw*s); ch = Math.floor(ch*s);
            }
            c.width = cw; c.height = ch;
            c.getContext(\"2d\").drawImage(v, 0, 0, cw, ch);
            return c.toDataURL(\"image/jpeg\", 0.95);
        })()
    \"\"\")
    img = Image.open(io.BytesIO(b64decode(d.split(\",\")[1]))).convert(\"RGB\")
    MAX_INPUT = 640
    w, h = img.size
    if max(w, h) > MAX_INPUT:
        scale = MAX_INPUT / max(w, h)
        img = img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
    return img

def stop_camera():
    return eval_js(\"\"\"
        (async () => {
            const v = document.getElementById(\"webcam\");
            if (v && v.srcObject) v.srcObject.getTracks().forEach(t => t.stop());
            const c = document.getElementById(\"cam-container\"); if (c) c.remove();
            return \"stopped\";
        })()
    \"\"\")

def to_html(img, w=384):
    b = io.BytesIO(); img.save(b, format='JPEG', quality=88)
    return f'<img src=\"data:image/jpeg;base64,{b64encode(b.getvalue()).decode()}\" width=\"{w}\"/>'

# --- Face detector ---
_face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def _detect_faces(pil_img):
    """All faces sorted largest-first."""
    gray = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2GRAY)
    faces = _face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60, 60))
    if len(faces) == 0:
        return []
    return sorted(faces, key=lambda f: f[2]*f[3], reverse=True)

# --- HYBRID PIPELINE ---
def toonforge_process(pil_img, exstyle, style_degree, style_name, padding):
    \"\"\"Full hybrid pipeline: SceneNet body/bg + FaceForge faces.\"\"\"\n",
    all_faces = _detect_faces(pil_img)
    scene_full = scene_render(pil_img)

    if len(all_faces) == 0:
        return scene_full, 'scene_only(0 faces)'

    # Preserve original faces in SceneNet output
    orig_np = np.array(pil_img)
    scene_np = np.array(scene_full.resize(pil_img.size, Image.LANCZOS))
    mask_all = np.zeros(orig_np.shape[:2], dtype=np.float32)

    for (x, y, w, h) in all_faces:
        ex, ey = int(w * 0.3), int(h * 0.3)
        fx1, fy1 = max(0, x - ex), max(0, y - ey)
        fx2 = min(orig_np.shape[1], x + w + ex)
        fy2 = min(orig_np.shape[0], y + h + ey)
        mask_all[fy1:fy2, fx1:fx2] = 1.0

    ksize = max(31, int(min(all_faces[0][2], all_faces[0][3]) * 0.4) | 1)
    mask_all = cv2.GaussianBlur(mask_all, (ksize, ksize), 0)[:, :, np.newaxis]
    blended = (orig_np * mask_all + scene_np * (1 - mask_all)).astype(np.uint8)
    result_np = blended.copy()

    # FaceForge each face + composite
    n_done = 0
    for i, (x, y, w, h) in enumerate(all_faces):
        pad_x, pad_y = int(w * 0.8), int(h * 0.8)
        cx1, cy1 = max(0, x - pad_x), max(0, y - pad_y)
        cx2 = min(orig_np.shape[1], x + w + pad_x)
        cy2 = min(orig_np.shape[0], y + h + pad_y)
        face_crop = pil_img.crop((cx1, cy1, cx2, cy2))

        tmp_path = f'/tmp/_toonforge_face_{i}.jpg'
        face_crop.save(tmp_path, quality=95)
        aligned, instyle, msg = face_engine.detect_and_align_image(tmp_path, *padding)

        if aligned is None:
            continue

        vt_result, msg2 = face_engine.image_toonify(aligned, instyle, exstyle, style_degree, style_name)
        if vt_result is None:
            continue

        vt_pil = Image.fromarray(vt_result)

        # Elliptical feathered composite
        expand_x, expand_y = int(w * 0.45), int(h * 0.55)
        fx1, fy1 = max(0, x - expand_x), max(0, y - expand_y)
        fx2 = min(result_np.shape[1], x + w + expand_x)
        fy2 = min(result_np.shape[0], y + h + expand_y)
        rw, rh = fx2 - fx1, fy2 - fy1

        if rw < 10 or rh < 10:
            continue

        vt_resized = vt_pil.resize((rw, rh), Image.LANCZOS)
        vt_np = np.array(vt_resized)

        mask = np.zeros((rh, rw), dtype=np.float32)
        cx, cy = rw // 2, rh // 2
        ax, ay = int(rw * 0.42), int(rh * 0.45)
        cv2.ellipse(mask, (cx, cy), (ax, ay), 0, 0, 360, 1.0, -1)
        ks = max(31, int(min(rw, rh) * 0.3) | 1)
        mask = cv2.GaussianBlur(mask, (ks, ks), 0)[:, :, np.newaxis]

        bg = result_np[fy1:fy2, fx1:fx2].astype(np.float32)
        fg = vt_np.astype(np.float32)
        result_np[fy1:fy2, fx1:fx2] = (fg * mask + bg * (1 - mask)).astype(np.uint8)
        n_done += 1

    mode = f'hybrid({n_done}/{len(all_faces)} faces)' if n_done > 0 else f'scene_only({len(all_faces)} detected, 0 styled)'
    return Image.fromarray(result_np), mode

print('\u2705 ToonForge pipeline ready (SceneNet body/bg + FaceForge face)')


In [ ]:
# 5. 📸 Single photo test
f = grab_frame()

t0 = time.time()
result, mode = toonforge_process(f, exstyle, STYLE_DEGREE, STYLE, FACE_PADDING)
t_total = time.time() - t0

display(HTML(f'''
<div style="display:flex;gap:12px;font-family:monospace;">
  <div style="text-align:center"><div style="font-size:12px;margin-bottom:4px">📷 Original</div>{to_html(f, 400)}</div>
  <div style="text-align:center"><div style="font-size:12px;margin-bottom:4px">🎨 ToonForge ({mode})</div>{to_html(result, 400)}</div>
</div>'''))
mem = torch.cuda.memory_allocated() / 1024**3
print(f'Mode: {mode} | {t_total*1000:.0f}ms | GPU: {mem:.1f}GB')


In [ ]:
# 6. 📺 LIVE PREVIEW (multi-face) — Press STOP button to end

PRESET = 'balanced'
#   'fast'      →  ~2-3 FPS, lower quality
#   'balanced'  →  ~1-2 FPS, good quality
#   'quality'   →  ~0.5-1 FPS, maximum fidelity

presets = {
    'fast':     {'display_w': 300, 'jpg_q': 75},
    'balanced': {'display_w': 340, 'jpg_q': 82},
    'quality':  {'display_w': 380, 'jpg_q': 88},
}
P = presets[PRESET]

def to_html_fast(img, w):
    b = io.BytesIO(); img.save(b, format='JPEG', quality=P['jpg_q'])
    return f'<img src="data:image/jpeg;base64,{b64encode(b.getvalue()).decode()}" width="{w}"/>'

print(f'ToonForge Live: {PRESET} | {STYLE} (degree={STYLE_DEGREE})\n')
fc = 0
h = display(HTML('Warming up...'), display_id=True)

try:
    while True:
        t0 = time.time()
        f = grab_frame()
        t_grab = time.time() - t0

        t1 = time.time()
        result, mode = toonforge_process(f, exstyle, STYLE_DEGREE, STYLE, FACE_PADDING)
        t_proc = time.time() - t1

        torch.cuda.synchronize()
        fps = 1.0 / (time.time() - t0 + 1e-6)
        fc += 1
        mem = torch.cuda.memory_allocated() / 1024**3
        dw = P['display_w']

        h.update(HTML(f"""
        <div style="display:flex;gap:8px;align-items:center;font-family:monospace;">
          <div style="text-align:center"><div style="font-size:11px;margin-bottom:3px">Original</div>{to_html_fast(f, dw)}</div>
          <div style="text-align:center"><div style="font-size:11px;margin-bottom:3px">ToonForge</div>{to_html_fast(result, dw)}</div>
        </div>
        <div style="font-family:monospace;font-size:10px;color:#888;margin-top:4px">
          #{fc} | {fps:.1f}fps | {mode} | grab:{t_grab*1000:.0f} proc:{t_proc*1000:.0f}ms | GPU:{mem:.1f}GB
        </div>"""))

except KeyboardInterrupt:
    print(f'\n{fc} frames processed. Peak GPU: {torch.cuda.max_memory_allocated()/1024**3:.1f}GB')


In [ ]:
# 7. 🔄 Switch style on the fly
#   cartoon1-d through cartoon5-d
#   arcane1-d, arcane2-d  |  pixar1-d  |  comic1-d, comic2-d
#   caricature1-d, caricature2-d
#   (drop -d for fixed-style variants)

STYLE = 'arcane1-d'     # ← change this
STYLE_DEGREE = 0.6      # ← adjust 0.0-1.0

exstyle, info = face_engine.load_model(STYLE)
print(info)
print('Re-run cell 6 for live preview with new style')


In [ ]:
# 8. 🎭 Compare 5 styles on one frame
f = grab_frame()

test_styles = ['cartoon1-d', 'arcane1-d', 'pixar1-d', 'comic1-d', 'caricature1-d']
cards = f'<div style="text-align:center"><div style="font-size:10px;margin-bottom:3px">📷 Original</div>{to_html(f, 180)}</div>'

# SceneNet only (no face styling)
scene_only = scene_render(f)
cards += f'<div style="text-align:center"><div style="font-size:10px;margin-bottom:3px">🖌 SceneNet Only</div>{to_html(scene_only, 180)}</div>'
print('  \u2705 SceneNet')

# Hybrid for each FaceForge style
for s in test_styles:
    ex, _ = face_engine.load_model(s)
    result, mode = toonforge_process(f, ex, 0.5, s, FACE_PADDING)
    cards += f'<div style="text-align:center"><div style="font-size:10px;margin-bottom:3px">🎨 {s}</div>{to_html(result, 180)}</div>'
    print(f'  \u2705 {s} ({mode})')

display(HTML(f'<div style="display:flex;gap:4px;flex-wrap:wrap;font-family:monospace">{cards}</div>'))

# Reset to active style
exstyle, _ = face_engine.load_model(STYLE)
print(f'\nReset to: {STYLE}')


In [ ]:
# 9. Release camera
print(stop_camera())
